# SHAP explainability report

## Scientific question

Which input dimensions contribute to held-out predictions, and are those attributions interpretable within the selected model representation?

## Rationale

SHAP is computed from training-only backgrounds and held-out samples. Attribution magnitudes are compared within a model, not across incompatible input spaces.

In [ ]:

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

from br_analysis.config import ProjectPaths
from br_analysis.explanations import explanation_stability, load_explanation
from br_analysis.registry import get_model_spec
from br_analysis.scientific_reports import (
    aggregate_metrics,
    configuration_table,
    load_saved_run,
    prediction_diagnostics,
)
from br_analysis.tasks import get_task

In [ ]:
TASK_ID = "c_vs_ip"
MODEL_ID = "eegnet_agnostic"
paths = ProjectPaths.from_env("explainability", create_dirs=False)
task = get_task(TASK_ID)
model_spec = get_model_spec(MODEL_ID)
run = load_saved_run(paths.artifacts_dir, TASK_ID, MODEL_ID)
explanation_path = paths.artifacts_dir / TASK_ID / MODEL_ID / "shap_values.npz"
explanation = load_explanation(explanation_path) if explanation_path.is_file() else None

In [ ]:
display(Markdown("## Selected task, model and reproducible command"))
display(configuration_table(task, model_spec))
command = f"br-analysis run --task {TASK_ID} --model {MODEL_ID} --explain"
display(Markdown(f"```powershell\n{command}\n```"))
if run is not None:
    display(aggregate_metrics(run))
    display(prediction_diagnostics(run))

## Attribution diagnostics and model-specific visualisation

In [ ]:
if explanation is None:
    print("No SHAP artifact. Run the command above.")
else:
    values = np.asarray(explanation.values)
    mean_abs = np.mean(np.abs(values), axis=tuple(range(values.ndim - 1)))
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(np.arange(len(mean_abs)), mean_abs)
    ax.set(xlabel="Flattened input dimension", ylabel="Mean absolute SHAP value")
    plt.show()
    midpoint = len(values) // 2
    if midpoint > 0:
        stability = explanation_stability(
            type(explanation)(values[:midpoint], explanation.base_values, explanation.samples[:midpoint], explanation.class_ids, explanation.method, explanation.feature_names),
            type(explanation)(values[midpoint:], explanation.base_values, explanation.samples[midpoint:], explanation.class_ids, explanation.method, explanation.feature_names),
        )
        print(f"Split-half attribution stability: {stability:.3f}")

## Interpretation, limitations and baseline comparison

Attributions describe model sensitivity, not causal neural mechanisms. Check stability, class specificity and consistency with simpler baselines. Raw SHAP magnitudes must not be compared directly between covariance, tabular and tensor representations.